# Data merge and feature engineering


## I. Importing essential libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
import json

## II. Impoting datasets and creating base dataframes

In [2]:
data_dir = Path("../data")
weather_file = data_dir / "weather_processed.csv"
alarms_file = data_dir / "war_events_processed.csv"
isw_file = data_dir / "isw_processed_svd.csv"
tg_file = data_dir / "telegram_processed_svd.csv"

output = "final_merged_dataset.parquet"

In [3]:
df_weather = pd.read_csv(weather_file)
df_war_events = pd.read_csv(alarms_file)
df_isw_matrix = pd.read_csv(isw_file)
df_tg_matrix = pd.read_csv(tg_file)

## III. Data merge

### Weather (main) and War_events merge

In [4]:
df_weather.shape

(844417, 40)

In [5]:
df_war_events.shape

(924664, 5)

In [6]:
df_weather.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.300,100.0,4.17,...,True,False,False,3,0,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька
1,2022-02-24 00:00:00,3,4.9,0.7,2.6,0.0,83.7,0.118,100.0,4.17,...,True,False,False,3,0,Lutsk,2022-02-24,07:13:36,17:51:06,Волинська
2,2022-02-24 00:00:00,4,8.0,-2.0,3.1,-2.2,70.6,0.000,0.0,0.00,...,False,False,False,3,0,Dnipro,2022-02-24,06:31:28,17:15:39,Дніпропетровська
3,2022-02-24 00:00:00,5,5.6,1.9,3.6,0.7,81.0,1.300,100.0,20.83,...,True,False,False,3,0,Donetsk,2022-02-24,06:19:37,17:05:06,Донецька
4,2022-02-24 00:00:00,6,5.5,1.5,3.3,0.2,80.8,0.800,100.0,4.17,...,True,False,False,3,0,Zhytomyr,2022-02-24,06:59:29,17:38:29,Житомирська


In [7]:
df_war_events.head()

,datetime_hour,region_id,region_key,alarm_minutes_in_hour,alarm_active
0,2022-02-24 00:00:00,1,АР Крим,0.0,0
1,2022-02-24 00:00:00,2,Вінницька,0.0,0
2,2022-02-24 00:00:00,3,Волинська,0.0,0
3,2022-02-24 00:00:00,4,Дніпропетровська,0.0,0
4,2022-02-24 00:00:00,5,Донецька,0.0,0


In [8]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [9]:
df_final = pd.merge(df_weather, df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], on=['datetime_hour', 'region_id'], how='left')

In [10]:
df_final.shape

(844417, 42)

In [11]:
df_final['region_key'].unique()

<ArrowStringArray>
[        'Вінницька',         'Волинська',  'Дніпропетровська',
          'Донецька',       'Житомирська',      'Закарпатська',
        'Запорізька', 'Івано-Франківська',          'Київська',
    'Кіровоградська',         'Львівська',      'Миколаївська',
           'Одеська',        'Полтавська',        'Рівненська',
           'Сумська',     'Тернопільська',        'Харківська',
        'Херсонська',       'Хмельницька',         'Черкаська',
       'Чернівецька',      'Чернігівська',              'Київ']
Length: 24, dtype: str

#### We have 844417 rows, just as in main weather dataset, due to the lack of data on occupied territory - Crimea and Luhansk. 

In [12]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,False,3,0,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
24,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,False,3,1,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
48,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,False,3,2,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
72,2022-02-24 03:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,False,3,3,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
96,2022-02-24 04:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,False,3,4,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0


In [13]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 844417 entries, 0 to 844416
Data columns (total 42 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   datetime_hour                  844417 non-null  datetime64[us]
 1   region_id                      844417 non-null  int64         
 2   day_tempmax                    844417 non-null  float64       
 3   day_tempmin                    844417 non-null  float64       
 4   day_temp                       844417 non-null  float64       
 5   day_dew                        844417 non-null  float64       
 6   day_humidity                   844417 non-null  float64       
 7   day_precip                     844417 non-null  float64       
 8   day_precipprob                 844417 non-null  float64       
 9   day_precipcover                844417 non-null  float64       
 10  day_windgust                   844417 non-null  float64       
 11  day_cloudcover  

### ISW Merge

In [14]:
df_isw_matrix.shape

(1470, 151)

In [15]:
df_isw_matrix.head()

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24,0.648733,-0.078386,0.180873,0.059880,0.070008,-0.042071,-0.006712,0.006174,-0.014324,...,0.013046,0.010118,-0.017252,-0.029334,-0.010031,-0.020943,0.039399,-0.023117,0.000179,-0.014684
1,2022-02-24,0.579583,-0.092636,-0.024936,0.148737,0.067631,-0.022840,0.013561,0.027065,-0.037442,...,-0.011816,0.006371,-0.015602,-0.023692,-0.022085,-0.009336,0.000669,-0.016102,0.022940,-0.018549
2,2022-02-25,0.645442,-0.072671,0.238645,0.059066,0.105737,-0.047058,0.013866,0.010163,-0.004477,...,0.031812,0.024702,0.003920,-0.017590,0.012570,-0.017708,0.007496,-0.010090,-0.014617,-0.008044
3,2022-02-26,0.656647,-0.083867,0.262991,0.087335,0.116642,-0.047191,0.013102,0.010315,-0.007486,...,0.011975,-0.003639,-0.008046,-0.007291,-0.002018,-0.003440,0.022464,0.000583,-0.007783,0.010795
4,2022-02-27,0.689454,-0.069918,0.259447,0.081355,0.116101,-0.040898,0.010741,0.011474,-0.003294,...,-0.009857,0.001316,-0.016979,0.002200,0.000768,0.010710,-0.005216,0.010155,-0.010394,0.002149


In [16]:
df_final.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,False,3,0,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
24,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,False,3,1,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
48,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,False,3,2,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
72,2022-02-24 03:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,False,3,3,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
96,2022-02-24 04:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,False,3,4,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0


In [17]:
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [18]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date

In [19]:
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [20]:
df_isw_matrix = df_isw_matrix.fillna(0)

In [21]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [22]:
df_final.head(15)

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.013046,0.010118,-0.017252,-0.029334,-0.010031,-0.020943,0.039399,-0.023117,0.000179,-0.014684
1,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,-0.011816,0.006371,-0.015602,-0.023692,-0.022085,-0.009336,0.000669,-0.016102,0.022940,-0.018549
2,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.013046,0.010118,-0.017252,-0.029334,-0.010031,-0.020943,0.039399,-0.023117,0.000179,-0.014684
3,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,-0.011816,0.006371,-0.015602,-0.023692,-0.022085,-0.009336,0.000669,-0.016102,0.022940,-0.018549
4,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.013046,0.010118,-0.017252,-0.029334,-0.010031,-0.020943,0.039399,-0.023117,0.000179,-0.014684
5,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,-0.011816,0.006371,-0.015602,-0.023692,-0.022085,-0.009336,0.000669,-0.016102,0.022940,-0.018549
6,2022-02-24 03:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.013046,0.010118,-0.017252,-0.029334,-0.010031,-0.020943,0.039399,-0.023117,0.000179,-0.014684
7,2022-02-24 03:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,-0.011816,0.006371,-0.015602,-0.023692,-0.022085,-0.009336,0.000669,-0.016102,0.022940,-0.018549
8,2022-02-24 04:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.013046,0.010118,-0.017252,-0.029334,-0.010031,-0.020943,0.039399,-0.023117,0.000179,-0.014684
9,2022-02-24 04:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,-0.011816,0.006371,-0.015602,-0.023692,-0.022085,-0.009336,0.000669,-0.016102,0.022940,-0.018549


In [23]:
df_final.shape

(844993, 192)

In [24]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 844993 entries, 0 to 844992
Columns: 192 entries, datetime_hour to isw_topic_149
dtypes: bool(4), datetime64[us](1), float64(178), int64(4), object(1), str(4)
memory usage: 1.2+ GB


### Telegram Merge

In [25]:
df_tg_matrix.shape

(130418, 252)

In [26]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2026-03-17 01:31:21,kpszsu,0.138465,-0.060347,0.284671,0.035601,0.116456,-0.045463,-0.021404,0.031473,...,0.032808,0.093800,-0.069880,0.004953,-0.050572,0.002973,0.033703,-0.030074,-0.039642,0.065714
1,2026-03-17 01:21:32,kpszsu,0.081445,-0.029977,0.247861,0.054075,0.095709,-0.057322,-0.032291,-0.002144,...,0.022844,0.061741,-0.099778,-0.040678,-0.017302,0.050442,-0.032312,-0.019222,0.001714,-0.011180
2,2026-03-17 00:45:16,kpszsu,0.105084,-0.046298,0.205981,0.022232,0.084708,-0.026968,-0.018273,0.035228,...,-0.006270,-0.004745,-0.021264,-0.040773,0.023039,0.027518,-0.034993,0.000479,0.020928,-0.028859
3,2026-03-17 00:33:10,kpszsu,0.157535,-0.071472,0.326986,0.093080,0.102398,-0.106918,-0.002767,0.013840,...,-0.002598,0.002334,-0.003293,-0.005758,0.003161,0.003687,-0.003392,-0.015742,0.024436,0.006963
4,2026-03-16 23:30:39,kpszsu,0.086284,-0.013447,0.206107,0.011940,0.060368,-0.059965,-0.001821,0.019170,...,0.000333,0.018163,-0.014486,0.006608,-0.000406,0.008269,0.006597,-0.000130,0.009959,0.008867


In [27]:
df_tg_matrix.tail(5)

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
130413,2022-02-25 09:45:10,kpszsu,0.020260,0.003010,0.019604,-0.004270,-0.017974,0.042540,-0.001457,0.009154,...,0.027771,0.011515,0.030133,0.006338,-0.007921,0.002182,0.009057,-0.025187,-0.000008,0.004170
130414,2022-02-25 09:43:53,kpszsu,0.036856,-0.017407,0.025895,0.005321,-0.041815,0.076052,-0.054188,0.138157,...,0.015877,0.010110,0.036470,-0.015028,-0.006703,0.047245,0.009053,-0.006635,-0.021697,-0.023936
130415,2022-02-25 00:43:47,kpszsu,0.025424,-0.008920,0.037626,0.004519,-0.031715,0.075999,-0.046593,0.148629,...,-0.002928,0.014550,0.020617,-0.005018,-0.016592,0.018170,0.012201,-0.006091,-0.014534,-0.011366
130416,2022-02-25 00:43:23,kpszsu,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
130417,2022-02-24 23:17:22,kpszsu,0.010204,0.001464,0.024673,0.001217,-0.015579,0.039516,-0.024036,0.093166,...,0.010781,0.009812,0.062213,-0.022939,-0.013145,0.059785,0.038512,0.020110,-0.001856,0.013671


In [28]:
df_tg_matrix['datetime_hour'] = pd.to_datetime(df_tg_matrix['date']).dt.floor('h')

C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\1220961098.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tg_matrix['datetime_hour'] = pd.to_datetime(df_tg_matrix['date']).dt.floor('h')


In [29]:
topic_cols = [c for c in df_tg_matrix.columns if 'tg_topic_' in c]
tg_hourly = (df_tg_matrix.groupby('datetime_hour')[topic_cols].mean().sort_index().reset_index())

C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\1074555927.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tg_hourly = (df_tg_matrix.groupby('datetime_hour')[topic_cols].mean().sort_index().reset_index())


In [30]:
tg_hourly.head()

,datetime_hour,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2022-02-24 05:00:00,0.007571,0.001841,0.017769,0.000372,-0.018203,0.047323,-0.014073,0.046282,-0.004875,...,0.008067,0.004332,-0.013217,-0.005994,0.011600,0.003181,-0.003585,-0.001433,0.005914,-0.003696
1,2022-02-24 06:00:00,0.005030,-0.000197,0.014174,0.001412,-0.006449,0.015642,-0.010847,0.042765,0.003864,...,-0.003696,-0.006636,-0.002343,0.009105,0.008887,-0.002037,0.001363,0.010417,-0.006575,0.004531
2,2022-02-24 07:00:00,0.003351,-0.000047,0.009519,0.000819,-0.005333,0.013872,-0.009793,0.035888,0.005566,...,0.008899,0.002575,0.002375,0.018925,0.000053,-0.004988,0.001595,-0.006826,-0.006113,0.000061
3,2022-02-24 08:00:00,0.027905,0.001534,0.040455,-0.002476,-0.021672,0.060173,-0.014146,0.057490,0.018659,...,0.001277,-0.000130,0.012205,0.002413,0.000554,-0.005761,-0.001243,0.009363,-0.006740,-0.003641
4,2022-02-24 09:00:00,0.015303,0.001760,0.027612,0.000139,-0.017373,0.044288,-0.019291,0.077023,0.015139,...,-0.006704,0.012909,-0.005992,0.001899,0.005360,-0.002592,-0.006409,0.000631,-0.007607,0.004281


In [31]:
all_hours = pd.DataFrame({'datetime_hour': pd.date_range(df_final['datetime_hour'].min(), df_final['datetime_hour'].max(), freq='h')})
tg_hourly = all_hours.merge(tg_hourly, on='datetime_hour', how='left').fillna(0)

In [32]:
tg_hourly.head(10)

,datetime_hour,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2022-02-24 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2022-02-24 01:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2022-02-24 02:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2022-02-24 03:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2022-02-24 04:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,2022-02-24 05:00:00,0.007571,0.001841,0.017769,0.000372,-0.018203,0.047323,-0.014073,0.046282,-0.004875,...,0.008067,0.004332,-0.013217,-0.005994,0.011600,0.003181,-0.003585,-0.001433,0.005914,-0.003696
6,2022-02-24 06:00:00,0.005030,-0.000197,0.014174,0.001412,-0.006449,0.015642,-0.010847,0.042765,0.003864,...,-0.003696,-0.006636,-0.002343,0.009105,0.008887,-0.002037,0.001363,0.010417,-0.006575,0.004531
7,2022-02-24 07:00:00,0.003351,-0.000047,0.009519,0.000819,-0.005333,0.013872,-0.009793,0.035888,0.005566,...,0.008899,0.002575,0.002375,0.018925,0.000053,-0.004988,0.001595,-0.006826,-0.006113,0.000061
8,2022-02-24 08:00:00,0.027905,0.001534,0.040455,-0.002476,-0.021672,0.060173,-0.014146,0.057490,0.018659,...,0.001277,-0.000130,0.012205,0.002413,0.000554,-0.005761,-0.001243,0.009363,-0.006740,-0.003641
9,2022-02-24 09:00:00,0.015303,0.001760,0.027612,0.000139,-0.017373,0.044288,-0.019291,0.077023,0.015139,...,-0.006704,0.012909,-0.005992,0.001899,0.005360,-0.002592,-0.006409,0.000631,-0.007607,0.004281


In [33]:
df_final = df_final.merge(tg_hourly, on="datetime_hour", how="left")

In [34]:
df_final.shape

(844993, 442)

In [35]:
df_final.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [36]:
df_final.tail()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
844988,2026-03-16 19:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,...,-0.022194,0.005554,0.021317,-0.014790,-0.004291,0.007044,0.008297,-0.009643,-0.006328,0.022611
844989,2026-03-16 20:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,...,-0.001465,0.000396,0.008035,0.005693,0.004914,0.000984,-0.004285,0.004516,-0.009363,0.002848
844990,2026-03-16 21:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,...,0.000952,-0.007326,0.006638,-0.001482,0.000911,-0.002082,-0.005492,-0.002772,0.002854,-0.007854
844991,2026-03-16 22:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,...,0.031434,-0.035452,-0.023091,0.020039,0.024311,-0.015323,-0.012948,0.021162,-0.025514,0.002333
844992,2026-03-16 23:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,...,-0.027735,0.009093,0.019108,-0.019050,0.014821,0.004074,0.003833,0.002178,0.032295,-0.021737


In [37]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 844993 entries, 0 to 844992
Columns: 442 entries, datetime_hour to tg_topic_249
dtypes: bool(4), datetime64[us](1), float64(428), int64(4), object(1), str(4)
memory usage: 2.8+ GB


## IV. Feature engineering

### Alarms & weather

In [38]:
df_final = df_final.sort_values(["region_id", "datetime_hour"])

In [39]:
df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)

lag_cols = ["alarm_lag_1", "alarm_lag_3", "alarm_lag_6", "alarm_lag_12"]
df_final[lag_cols] = df_final[lag_cols].fillna(0)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\3164348077.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\3164348077.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\3164348077.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of

In [40]:
df_final['alarms_in_last_24h'] = df_final.groupby('region_id')['alarm_active'].transform(lambda x: x.shift(1).rolling(24, min_periods=1).sum())
df_final['alarms_in_last_24h'] = df_final['alarms_in_last_24h'].fillna(0)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\2630719587.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['alarms_in_last_24h'] = df_final.groupby('region_id')['alarm_active'].transform(lambda x: x.shift(1).rolling(24, min_periods=1).sum())


In [41]:
df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\4068364582.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\4068364582.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)


In [42]:
hourly_total = df_final.groupby('datetime_hour')['alarm_active'].sum().shift(1)
df_final['total_active_alarms_lag1'] = df_final['datetime_hour'].map(hourly_total)
df_final['total_active_alarms_lag1'] = df_final['total_active_alarms_lag1'].fillna(0)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\1840295347.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['total_active_alarms_lag1'] = df_final['datetime_hour'].map(hourly_total)


In [43]:
neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(index='datetime_hour', columns='region_id', values='alarm_active', fill_value=0)

neighbour_alarm_matrix = pd.DataFrame(index=alarms_matrix.index)

for region, neighbours in neighbouring_regions.items():
    valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]
    
    if valid_neighbours:
        neighbour_alarm_matrix[region] = alarms_matrix[valid_neighbours].sum(axis=1)
    else:
        neighbour_alarm_matrix[region] = 0

neighbour_alarm_matrix = neighbour_alarm_matrix.shift(1)
neighbour_alarm_long = (neighbour_alarm_matrix.stack().reset_index())
neighbour_alarm_long.columns = ['datetime_hour','region_id','neighbour_alarms']
df_final = df_final.merge(neighbour_alarm_long,on=['datetime_hour', 'region_id'], how='left')
df_final['neighbour_alarms'] = df_final['neighbour_alarms'].fillna(0)

In [44]:
def hours_since_last_alarm_vectorized(series):
    shifted = series.shift(1).fillna(0)
    alarm_cumsum = shifted.cumsum()
    result = shifted.groupby(alarm_cumsum).cumcount()
    return result

df_final['hours_since_last_alarm'] = (df_final.groupby('region_id')['alarm_active'].transform(hours_since_last_alarm_vectorized))

C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\2225684187.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['hours_since_last_alarm'] = (df_final.groupby('region_id')['alarm_active'].transform(hours_since_last_alarm_vectorized))


In [45]:
df_final.head(10)

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,0
1,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,1
2,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,2
3,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,3
4,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,4
5,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,5
6,2022-02-24 03:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,6
7,2022-02-24 03:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,7
8,2022-02-24 04:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,8
9,2022-02-24 04:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,9


In [46]:
df_final.isna().sum()

datetime_hour               0
region_id                   0
day_tempmax                 0
day_tempmin                 0
day_temp                    0
                           ..
is_weekend                  0
is_night                    0
total_active_alarms_lag1    0
neighbour_alarms            0
hours_since_last_alarm      0
Length: 452, dtype: int64

In [47]:
df_final.tail()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm
844988,2026-03-16 19:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,0,0,8.0,1.0,7
844989,2026-03-16 20:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,3.0,0,0,11.0,1.0,8
844990,2026-03-16 21:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,3.0,0,0,8.0,0.0,9
844991,2026-03-16 22:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,0,0,9.0,0.0,10
844992,2026-03-16 23:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,4.0,0,1,10.0,1.0,0


### ISW

In [48]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_final[topic_cols] = df_final[topic_cols].fillna(0)

df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final['isw_topic_mean'] = df_isw_abs.mean(axis=1)
df_final['isw_topic_entropy'] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\1786420544.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\1786420544.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\1786420544.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

In [49]:
df_final['isw_velocity_24h'] = (df_final.groupby('region_id')['isw_total_intensity'].diff(24).fillna(0))
df_final['isw_intensity_ema'] = (df_final.groupby('region_id')['isw_total_intensity'].transform(lambda x: x.shift(1).ewm(span=24).mean()))

isw_cols = ['isw_velocity_24h', 'isw_intensity_ema', 'isw_topic_entropy']
df_final[isw_cols] = df_final[isw_cols].fillna(0)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\3242989996.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_velocity_24h'] = (df_final.groupby('region_id')['isw_total_intensity'].diff(24).fillna(0))
C:\Users\Uw11\AppData\Local\Temp\ipykernel_11508\3242989996.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_intensity_ema'] = (df_final.groupby('region_id')['isw_total_intensity'].transform(lambda x: x.shift(1).ewm(span=24).mean()))


In [50]:
df_final = df_final.copy()

### TELEGRAM

In [51]:
df_final.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_topic_entropy,isw_velocity_24h,isw_intensity_ema
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0,6.866012,0.060253,0.648733,0.045773,4.569005,0.0,0.000000
1,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,1,7.045049,0.057999,0.579583,0.046967,4.576397,0.0,6.866012
2,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,2,6.866012,0.060253,0.648733,0.045773,4.569005,0.0,6.959261
3,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,3,7.045049,0.057999,0.579583,0.046967,4.576397,0.0,6.925553
4,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,4,6.866012,0.060253,0.648733,0.045773,4.569005,0.0,6.959261


In [52]:
tg_cols = [c for c in df_final.columns if 'tg_topic_' in c]
df_tg_abs = df_final[tg_cols].abs()

df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
df_final['tg_topic_max'] = df_tg_abs.max(axis=1)
df_final['tg_topic_entropy'] = -(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) * np.log(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

In [53]:
df_final['tg_velocity_3h'] = (df_final.groupby('region_id')['tg_total_intensity'].diff(3).fillna(0))
df_final['tg_intensity_ema_6h'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: x.ewm(span=6).mean()))
df_final['tg_intensity_zscore'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: (x - x.rolling(24, min_periods=1).mean()) / (x.rolling(24, min_periods=1).std() + 1e-9)))

In [54]:
tg_features_cols = [c for c in df_final.columns if ('tg_' in c and 'topic' not in c)]
df_final[tg_features_cols] = df_final[tg_features_cols].fillna(0)

In [55]:
df_final.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_entropy,isw_velocity_24h,isw_intensity_ema,tg_total_intensity,tg_topic_std,tg_topic_max,tg_topic_entropy,tg_velocity_3h,tg_intensity_ema_6h,tg_intensity_zscore
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,4.569005,0.0,0.000000,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
1,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,4.576397,0.0,6.866012,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
2,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,4.569005,0.0,6.959261,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
3,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,4.576397,0.0,6.925553,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
4,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,4.569005,0.0,6.959261,0.0,0.0,0.0,-0.0,0.0,0.0,0.0


# ТУТ БУДЕ EDA

## Data shift for further models training

In [56]:
df_to_train = df_final.copy()
df_to_train = df_to_train.sort_values(['region_id', 'datetime_hour'])

In [57]:
isw_cols = [c for c in df_to_train.columns if 'isw_' in c]
df_to_train[isw_cols] = df_to_train.groupby('region_id')[isw_cols].shift(24).fillna(0)

In [58]:
df_to_train.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_entropy,isw_velocity_24h,isw_intensity_ema,tg_total_intensity,tg_topic_std,tg_topic_max,tg_topic_entropy,tg_velocity_3h,tg_intensity_ema_6h,tg_intensity_zscore
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
1,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
2,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
3,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
4,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0


In [59]:
tg_cols = [c for c in df_to_train.columns if 'tg_' in c]
df_to_train[tg_cols] = df_to_train.groupby('region_id')[tg_cols].shift(1).fillna(0)

In [60]:
df_to_train.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_entropy,isw_velocity_24h,isw_intensity_ema,tg_total_intensity,tg_topic_std,tg_topic_max,tg_topic_entropy,tg_velocity_3h,tg_intensity_ema_6h,tg_intensity_zscore
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
2,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
3,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
4,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0


In [61]:
hour_weather_cols = [c for c in df_to_train.columns if c.startswith('hour_')]
for col in hour_weather_cols:
    if df_to_train[col].dtype == bool:
        df_to_train[col] = df_to_train.groupby('region_id')[col].shift(1).fillna(False)
    else:
        df_to_train[col] = df_to_train.groupby('region_id')[col].shift(1).fillna(0)

In [62]:
day_weather_cols = [c for c in df_to_train.columns if (c.startswith('day_') and c not in ['day_datetime', 'day_sunrise', 'day_sunset', 'day_moonphase', 'day_of_week'])]
df_to_train[day_weather_cols] = df_to_train.groupby('region_id')[day_weather_cols].shift(24).fillna(0)

In [63]:
df_to_train.isna().sum()

datetime_hour          0
region_id              0
day_tempmax            0
day_tempmin            0
day_temp               0
                      ..
tg_topic_max           0
tg_topic_entropy       0
tg_velocity_3h         0
tg_intensity_ema_6h    0
tg_intensity_zscore    0
Length: 466, dtype: int64

In [64]:
df_to_train['alarm_minutes_in_hour'] = (df_to_train.groupby('region_id')['alarm_minutes_in_hour'].shift(1).fillna(0))

In [65]:
df_to_train.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_entropy,isw_velocity_24h,isw_intensity_ema,tg_total_intensity,tg_topic_std,tg_topic_max,tg_topic_entropy,tg_velocity_3h,tg_intensity_ema_6h,tg_intensity_zscore
0,2022-02-24 00:00:00,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022-02-24 00:00:00,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
2,2022-02-24 01:00:00,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
3,2022-02-24 01:00:00,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0
4,2022-02-24 02:00:00,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0


In [66]:
print(f"Final dataset shape: {df_to_train.shape}")
print(f"Date range: {df_to_train['datetime_hour'].min()} → {df_to_train['datetime_hour'].max()}")
print(f"Regions: {df_to_train['region_id'].nunique()}")
print(f"Target balance: {df_to_train['alarm_active'].mean():.2%} positive")
print(f"NaN count: {df_to_train.isna().sum().sum()}")

Final dataset shape: (844993, 466)
Date range: 2022-02-24 00:00:00 → 2026-03-16 23:00:00
Regions: 24
Target balance: 20.64% positive
NaN count: 0


In [67]:
df_to_train.to_parquet(output, index=False, engine="pyarrow")